# Notebook 1: Data Exploration

Explore the synthetic social network dataset and understand its properties.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from src.dataset.loader import get_dataset
from src.dataset.preprocess import compute_graph_statistics, normalize_features

In [ ]:
# Generate dataset
data, labels, desc = get_dataset(
    dataset_name="synthetic",
    n_normal=800, n_anomaly=200, n_attacker=50,
    feature_dim=16, seed=42,
)
print(desc)

In [ ]:
# Graph statistics
stats = compute_graph_statistics(data)
for k, v in stats.items():
    print(f"{k}: {v}")

In [ ]:
# Class distribution
class_names = ["Normal", "Anomaly", "Attacker"]
counts = torch.bincount(labels)
plt.bar(class_names, counts.numpy())
plt.ylabel("Count")
plt.title("Class Distribution")
plt.show()

In [ ]:
# Degree distribution by class
degree = torch.bincount(data.edge_index[0], minlength=data.num_nodes).float()
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, (cls, name) in enumerate(zip([0, 1, 2], class_names)):
    mask = labels == cls
    axes[i].hist(degree[mask].numpy(), bins=30, alpha=0.7)
    axes[i].set_title(f"{name} (n={mask.sum().item()})")
    axes[i].set_xlabel("Degree")
plt.tight_layout()
plt.show()

In [ ]:
# Feature visualization (PCA)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
x_2d = pca.fit_transform(data.x.numpy())

plt.figure(figsize=(8, 6))
colors = ["blue", "red", "orange"]
for i, (cls, name) in enumerate(zip([0, 1, 2], class_names)):
    mask = labels == cls
    plt.scatter(x_2d[mask, 0], x_2d[mask, 1], c=colors[i], label=name, alpha=0.5)
plt.legend()
plt.title("Node Features (PCA 2D)")
plt.show()